# Federated learning with Flower: phase 1, simulation

Five clients train together a model that recognises clothes and shoes (Fashion-MNIST,
10 classes), **without ever sharing their images**. In each round, the server sends the
model to the clients, each one trains it on its own data and sends back only the
weights, then the server averages them.

Here the five clients are simulated on this machine. You launch every run with
Flower's own command, as you would in a terminal or a batch job:

```
flwr run app --stream --run-config "<settings>"
```

`flwr run` sends the app in `app/` and its settings to a **SuperLink**, the Flower
server process; here it is a local one that simulates the clients. Only the
`--run-config` string changes from one step to the next, never the code. `run-name`
only names the run in the dashboard.

The **dashboard** is a separate web page that watches the results: it picks up every
run on its own, round by round.

| Dashboard tab | What it shows |
|---|---|
| **Solo or together?** | The federated model versus each client trained alone, and versus fully centralised training |
| **The run** | The local models in each round, who took part, client drift |
| **Network** | The messages exchanged during a round, replayed as an animation |
| **Compare** | Several runs on one chart |

Each step: **predict**, run, check in the dashboard, **discuss** with your group. A run
takes 1 to 2 minutes; the first round takes 20 to 60 s, while the clients start.

## 0. Setup

In [ ]:
from flworkshop import viz

viz.setup()

Click **Open the dashboard** and keep the tab open next to this one.

## 1. First run

Each client holds only **2 classes** out of 10: one has only T-shirts and sneakers,
another only trousers and shirts, and so on.

> **Predict.** How many of the 10 classes can a client recognise alone? How many will
> the federated model recognise after 10 rounds?

Run it. The cell shows the server's log as it goes: each round samples the clients,
receives their results and aggregates them.

In [ ]:
!flwr run app --stream --run-config "run-name='pathological' partitioning='pathological'"

Open the **Solo or together?** tab. The "solo" and "centralised" references were
computed in advance with the same data and the same training budget.

In [ ]:
viz.dashboard("pathological")

The run did not live in this notebook: the SuperLink ran it, and keeps a record of it.
List the runs it knows, then open the **Network** tab to see what travelled.

In [ ]:
!flwr ls

In [ ]:
viz.dashboard("pathological", view="network")

> **Discuss**
> 1. How many classes does a client recognise alone? And the federated model?
> 2. In *What each model can recognise*, two classes are recognised by nobody, not even
>    the centralised model. Why? (Look at *Each client's data*.)
> 3. Does a client ever talk to another client? Find `Sampled`, `Received` and
>    `Aggregated` in the log above: which step of the animation does each match?
> 4. How many MB travelled in 10 rounds? Compare with the volume of the clients' data.

## 2. Heterogeneity

With **IID** data, each client holds a representative sample of all classes. With a
**Dirichlet** partitioning (α = 0.1), each client mostly holds a few classes, and some
clients hold far more images than others.

> **Predict.** Which run reaches the higher accuracy? Which one has the longer rounds,
> and why?

In [ ]:
!flwr run app --stream --run-config "run-name='iid' partitioning='iid'"

In [ ]:
!flwr run app --stream --run-config "run-name='dirichlet' partitioning='dirichlet' dirichlet-alpha=0.1"

In [ ]:
viz.dashboard("dirichlet", view="network")

> **Discuss**
> 1. *Compare* tab: rank the three runs. What makes federated learning hard?
> 2. *Network* tab, timeline of `dirichlet`: why does everyone wait for the same client?
>    Where have you seen this pattern on a cluster?
> 3. *The run* tab, *Client drift*: which client moves furthest from the global model?
>    What is special about its data?

## 3. TBD

To be determined.

## Going further

**Failures, simulated.** Only 2 clients out of 5 train in each round, or each client has
a one-in-two chance of dropping out. Look at *The run* tab: why does the curve go up
in a saw-tooth?

In [ ]:
!flwr run app --stream --run-config "run-name='participation_40' partitioning='dirichlet' dirichlet-alpha=0.1 fraction-train=0.4"

In [ ]:
!flwr run app --stream --run-config "run-name='dropouts_50' partitioning='dirichlet' dirichlet-alpha=0.1 dropout-rate=0.5"

**FedProx** penalises the distance between the local model and the global model
(`proximal-mu`). Does it beat FedAvg here? Look at *Client drift* in *The run* tab.

In [ ]:
# !flwr run app --stream --run-config "run-name='fedprox' partitioning='dirichlet' dirichlet-alpha=0.1 proximal-mu=0.1"

**Harder images: CIFAR-10** (planes, cars, animals…), if the CIFAR-10 data is present:

In [ ]:
# !flwr run app --stream --run-config "run-name='cifar' dataset='cifar10' partitioning='pathological'"

## Your turn

Every key below can go in the `--run-config` string; strings take single quotes.

| Key | Default | Role |
|---|---|---|
| `run-name` | run id | Name of the run in the dashboard |
| `partitioning` | `'iid'` | `'iid'`, `'dirichlet'` or `'pathological'` |
| `dirichlet-alpha` | 0.1 | Smaller = more unbalanced |
| `classes-per-partition` | 2 | Classes per client with `'pathological'` |
| `num-server-rounds` | 10 | Number of rounds |
| `local-epochs` | 2 | Local training epochs per round |
| `fraction-train` | 1.0 | Share of clients sampled in each round |
| `dropout-rate` | 0.0 | Probability that a client drops out during a round |
| `proximal-mu` | 0.0 | > 0 enables FedProx |
| `dataset` | `'fashion_mnist'` | `'fashion_mnist'` or `'cifar10'` |

The "solo / centralised" references exist only for the step 1 and 2 configurations.
A run you no longer want: interrupt the cell, then `!flwr stop <run id>` (the id is in
`!flwr ls`). `!flwr log <run id> --show` prints a run's log again.

In [ ]:
# !flwr run app --stream --run-config "run-name='my_try' partitioning='pathological' classes-per-partition=3"